# MRI_Detection on Google Colab
Brain tumour detection in MRI with YOLOv8m, compared with a version that adds CBAM attention and weighted feature fusion.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Waipo97/MRI-Image-Detection/blob/colab-ready/MRI_Detection/MRI_Detection_Colab.ipynb)

Before running:
1. Set the runtime to GPU (Runtime > Change runtime type > T4 GPU).
2. Keep the dataset zip somewhere in your Google Drive. The notebook looks for it by itself.
3. Run all cells. Results go to `MyDrive/MRI_Detection/runs/`.

If Colab disconnects, run the cells again. A finished training is skipped and an interrupted one continues from its last epoch.

## 0. Settings (edit here)

In [ ]:
DRIVE_FOLDER = "MRI_Detection"      # Drive folder where the results (runs/) are saved
EPOCHS       = None                 # None = use the value in configs/train.yaml
REPO_URL     = "https://github.com/Waipo97/MRI-Image-Detection.git"
BRANCH       = "colab-ready"     # change to "main" once the pull request is merged
# Only if you want to download from Kaggle instead of using the zip in Drive (needs a Kaggle token)
KAGGLE_DATASET = ""                 # "owner/dataset-name"

## 1. Setup

In [ ]:
import subprocess, sys, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "No GPU, change the runtime type")
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, pathlib, subprocess
DRIVE = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
(DRIVE / "runs").mkdir(parents=True, exist_ok=True)

# get the code
REPO = pathlib.Path("/content/MRI-Image-Detection")
if REPO.exists():
    for cmd in (["fetch", "origin", BRANCH], ["checkout", "-B", BRANCH, f"origin/{BRANCH}"]):
        subprocess.run(["git", "-C", str(REPO), *cmd], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
PROJ = REPO / "MRI_Detection"
os.chdir(PROJ)

# data on local disk, results on Drive
WORK = pathlib.Path("/content/mri_work"); WORK.mkdir(exist_ok=True)
os.environ.update(MRI_WORK_DIR=str(WORK), MRI_RUNS_DIR=str(DRIVE / "runs"),
                  MRI_RAW_ZIP=str(WORK / "data/raw/archive.zip"), MRI_WEIGHTS_DIR=str(DRIVE / "weights"))
print("project:", PROJ, "\nruns   :", os.environ["MRI_RUNS_DIR"])

In [ ]:
# torch is already installed on Colab, only ultralytics is needed (this exact version, the custom layers depend on it)
!pip -q install ultralytics==8.3.40

## 2. Data

In [ ]:
import shutil
zip_local = pathlib.Path(os.environ["MRI_RAW_ZIP"]); zip_local.parent.mkdir(parents=True, exist_ok=True)
zip_drive = DRIVE / "archive.zip"
if zip_local.exists():
    print("already on local disk")
elif KAGGLE_DATASET:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        from google.colab import files; files.upload()                     # upload kaggle.json
        os.makedirs("/root/.kaggle", exist_ok=True); shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle
    !kaggle datasets download -d {KAGGLE_DATASET} -p {zip_local.parent} && mv {zip_local.parent}/*.zip {zip_local}
else:
    # use MRI_Detection/archive.zip if it is there, otherwise look for a matching zip anywhere in Drive
    found = [zip_drive] if zip_drive.exists() else sorted(
        (p for p in pathlib.Path("/content/drive/MyDrive").rglob("*.zip")
         if any(k in p.name.lower() for k in ("archive", "mri", "brain", "tumor"))),
        key=lambda p: p.stat().st_size, reverse=True)
    if not found:
        raise FileNotFoundError("No dataset zip found in Drive. Upload it to My Drive and run this cell again.")
    print("candidates:", *[f"{p} ({p.stat().st_size/1e6:.0f} MB)" for p in found[:5]], sep="\n  ")
    shutil.copy(found[0], zip_local); print("using", found[0])
print(f"{zip_local.stat().st_size/1e6:.0f} MB")

In [ ]:
EP = f"--epochs {EPOCHS}" if EPOCHS else ""
# unzip, check the labels, drop bad images, split 70/20/10 and write the YOLO folders
!python -u scripts/run.py prepare

In [ ]:
import sys, yaml, random, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, str(PROJ / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config()
if EPOCHS: cfg["train"]["epochs"] = EPOCHS
P = md.get_paths(cfg); RUNS, YOLO_DIR, DATA_YAML = P["runs"], P["yolo_dir"], P["data_yaml"]
names = list(yaml.safe_load(open(DATA_YAML))["names"].values()); print("classes:", names)

print("excluded images:"); display(pd.read_csv(P["manifests"] / "excluded_images.csv"))
split = pd.read_csv(P["manifests"] / "split_manifest.csv")
display(pd.crosstab(split.folder, split.split)[["train", "val", "test"]])

random.seed(1)
files = random.sample(sorted((YOLO_DIR / "train" / "images").iterdir()), 8)
imgs = [viz.draw_boxes(f, viz.read_yolo_labels(YOLO_DIR / "train" / "labels" / (f.stem + ".txt")), names) for f in files]
viz.show_grid(imgs, ncols=4, figsize=(14, 3.6)); plt.show()

## 3. Baseline: YOLOv8m
Both models use `configs/train.yaml`, so the only difference between them is the architecture. A checkpoint is saved to Drive after every epoch.

In [ ]:
!python -u scripts/run.py baseline {EP}

## 4. Improved model: YOLOv8m + residual CBAM + weighted fusion

In [ ]:
!python -u scripts/run.py improved {EP}

In [ ]:
# if all the CBAM gammas stay close to 0, the attention is not really being used
from ultralytics import YOLO
from mri_detection import modules as cm
m = YOLO(str(RUNS / "proposed_yolov8m" / "weights" / "best.pt")).model
print("CBAM gamma    :", [round(float(b.gamma), 4) for b in m.modules() if isinstance(b, cm.CBAMAttention)])
print("Fusion weights:", [[round(float(w), 3) for w in b.w] for b in m.modules() if isinstance(b, cm.WeightedConcat)])

## 5. Results
Look at the validation table to compare the models. The test table is the one to report.

In [ ]:
import torch
DEVICE = md.config.resolve_device(cfg["train"]["device"])
res = evaluate.compare(RUNS, DATA_YAML, cfg["train"]["imgsz"], cfg["train"]["batch"], DEVICE)
for sp, (tab, pc) in res.items():
    print(f"\n===== {sp.upper()} ====="); display(tab.round(4)); display(pc.round(4))

In [ ]:
cb, cp = evaluate.load_curve(RUNS, "baseline_yolov8m"), evaluate.load_curve(RUNS, "proposed_yolov8m")
panels = [("train/box_loss", "Train box loss"), ("val/box_loss", "Val box loss"), ("metrics/mAP50(B)", "Val mAP@50"),
          ("metrics/mAP50-95(B)", "Val mAP@50-95"), ("metrics/precision(B)", "Val precision"), ("metrics/recall(B)", "Val recall")]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (col, t) in zip(ax.ravel(), panels):
    a.plot(cb["epoch"], cb[col], label="Baseline"); a.plot(cp["epoch"], cp[col], label="Improved")
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(RUNS / "training_curves_comparison.png", dpi=150); plt.show()

In [ ]:
from IPython.display import Image as IPImage
for n in ["baseline", "proposed"]:
    f = RUNS / f"{n}_test" / "confusion_matrix_normalized.png"
    print(n, "(test)"); display(IPImage(str(f), width=520)) if f.exists() else print("not found:", f)

In [ ]:
# ground truth, baseline and improved model on a few random test images
IMGSZ = cfg["train"]["imgsz"]
test_imgs = sorted((YOLO_DIR / "test" / "images").iterdir())
random.seed(3); picks = random.sample(test_imgs, min(6, len(test_imgs)))
mb = YOLO(str(RUNS / "baseline_yolov8m/weights/best.pt")); mp_ = YOLO(str(RUNS / "proposed_yolov8m/weights/best.pt"))
fig, ax = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)), squeeze=False)
for i, p in enumerate(picks):
    gt = viz.draw_boxes(p, viz.read_yolo_labels(YOLO_DIR / "test" / "labels" / (p.stem + ".txt")), names)
    pb = mb.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    pp = mp_.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    for j, (im, t) in enumerate(zip([gt, pb, pp], ["Ground truth", "Baseline", "Improved"])):
        ax[i, j].imshow(im); ax[i, j].set_title(t); ax[i, j].axis("off")
plt.tight_layout(); plt.savefig(RUNS / "qualitative_comparison.png", dpi=130); plt.show()

Notes for the report: look at mAP@50-95 and recall first, and only claim an improvement if the tables show it. There is one dataset, one seed and one split, and the dataset has no patient ids, so scores are probably higher than they would be on new patients. This is a research prototype, not a clinical tool.

## 6. Optional
### Ablation
Trains attention only and fusion only to see which part helps. It adds two more full trainings, so it is off by default.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    train.train_improved(cfg, DATA_YAML, RUNS, "ablation_attention_only", len(names), attention=True, weighted=False)
    train.train_improved(cfg, DATA_YAML, RUNS, "ablation_fusion_only", len(names), attention=False, weighted=True)
    ab = evaluate.compare(RUNS, DATA_YAML, IMGSZ, cfg["train"]["batch"], DEVICE, splits=("val",),
                          runs=(("baseline", "baseline_yolov8m"), ("attention_only", "ablation_attention_only"),
                                ("fusion_only", "ablation_fusion_only"), ("both", "proposed_yolov8m")))
    display(ab["val"][0].round(4)); ab["val"][0].round(4).to_csv(RUNS / "ablation_val.csv")

### Download the results

In [ ]:
out = shutil.make_archive("/content/mri_results", "zip", RUNS, ".")   # the same files are already on Drive
print(out)
from google.colab import files; files.download(out)